# Component 1 — Adaptive Web Attack Detection & Correlation Model
## Notebook 02: Data Cleaning, Feature Isolation & Train/Test Split

**Student:** Weerasinghe W.G.O.N (IT23378148)  
**Research Group:** J26-CS-303  
**Supervisor:** Mr. Amila Senarathna  
**Component:** Component 1 (Web Attack Detection & Correlation)  

---
### Objective of this Notebook:
1. Remove exact duplicate HTTP request records.
2. Remove incomplete core HTTP records.
3. Handle missing values in optional HTTP headers appropriately.
4. Isolate the 14 request-side detection features ($X$) from the 13 supervised CAPEC target labels ($y$).
5. Preserve the 5 contextual correlation attributes (`timestamp`, `src_ip`, `src_port`, `dst_ip`, `dst_port`) for Stage 2.
6. Create a reproducible 80:20 train/test split.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

# Add src to python path for modular imports
current_dir = Path.cwd()
if str(current_dir.parent) not in sys.path:
    sys.path.append(str(current_dir.parent))

from src.config import (
    RAW_DATA_FILE,
    PROCESSED_DATA_DIR,
    CORE_COLUMNS,
    CONTEXT_COLUMNS,
    DETECTION_FEATURES,
    MODEL_LABEL_COLUMNS,
    RARE_LABEL,
    RANDOM_STATE,
    TEST_SIZE,
    X_TRAIN_PATH,
    X_TEST_PATH,
    Y_TRAIN_PATH,
    Y_TEST_PATH,
    CONTEXT_TEST_PATH,
    CONTEXT_TEST_CSV
)

print("Environment initialized.")

Environment initialized.


## 1. Load Raw Dataset

We load the raw CSV file to perform all cleaning operations programmatically.

In [2]:
df = pd.read_csv(RAW_DATA_FILE, low_memory=False)
print(f"Raw dataset dimensions: {df.shape[0]:,} rows, {df.shape[1]} columns")

Raw dataset dimensions: 907,815 rows, 38 columns


## 2. Remove Exact Duplicate Records

Exact duplicates are removed to ensure identical records do not cross between training and testing sets.

In [3]:
rows_before = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = rows_before - len(df)

print(f"Duplicate records removed: {duplicates_removed:,}")
print(f"Records remaining: {len(df):,}")

Duplicate records removed: 2,746
Records remaining: 905,069


## 3. Remove Incomplete Core Request Records

Records missing mandatory HTTP components (method, protocol, URI, IP) cannot be used for request analysis.

In [4]:
rows_before_core = len(df)
df = df.dropna(subset=CORE_COLUMNS).copy()
core_removed = rows_before_core - len(df)

print(f"Incomplete core records removed: {core_removed}")
print(f"Final clean dataset size: {len(df):,} records")

Incomplete core records removed: 1
Final clean dataset size: 905,068 records


## 4. Handle Missing Values in Optional HTTP Headers

HTTP headers like `Referer`, `Cookie`, and `request_body` are frequently empty in normal requests. We fill these text fields with an empty string (`""`) so vectorizers can process them without errors.

In [5]:
optional_columns = [
    "request_user_agent", "request_referer", "request_origin",
    "request_cookie", "request_content_type", "request_accept",
    "request_accept_language", "request_accept_encoding",
    "request_connection", "request_body", "request_host"
]
df[optional_columns] = df[optional_columns].fillna("")

print(f"Missing values in detection features: {df[DETECTION_FEATURES].isnull().sum().sum()}")

Missing values in detection features: 0


## 5. Feature Isolation: Inputs ($X$), Targets ($y$), and Context (`context`)

- $X$: 14 request-side detection features
- $y$: 13 CAPEC multi-labels (excluding `248 - Command Injection` due to $N=1$)
- `context_df`: 5 contextual attributes for Stage 2 event correlation

In [6]:
X = df[DETECTION_FEATURES].copy()
y = df[MODEL_LABEL_COLUMNS].copy()
context_df = df[CONTEXT_COLUMNS].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"context_df shape: {context_df.shape}")

X shape: (905068, 14)
y shape: (905068, 13)
context_df shape: (905068, 5)


## 6. Reproducible Train / Test Split (80:20)

We split the data into 80% training (724,054 records) and 20% testing (181,014 records) using `random_state=42`. Crucially, we split `context_df` simultaneously to keep test event context directly aligned.

In [7]:
(X_train, X_test,
 y_train, y_test,
 context_train, context_test) = train_test_split(
    X, y, context_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape : {y_test.shape}")
print(f"context_test shape: {context_test.shape}")

X_train shape: (724054, 14)
X_test shape : (181014, 14)
y_train shape: (724054, 13)
y_test shape : (181014, 13)
context_test shape: (181014, 5)


### Label Representation across Splits

We verify that every single one of the 13 attack categories is properly represented in both training and testing partitions.

In [8]:
summary = pd.DataFrame({
    "Total Positives": y.sum(axis=0),
    "Train Positives": y_train.sum(axis=0),
    "Test Positives": y_test.sum(axis=0),
    "Test Ratio (%)": ((y_test.sum(axis=0) / y.sum(axis=0)) * 100).round(2)
})
display(summary.sort_values("Total Positives", ascending=False))

,Total Positives,Train Positives,Test Positives,Test Ratio (%)
000 - Normal,522734,418382,104352,19.96
66 - SQL Injection,250230,200121,50109,20.03
194 - Fake the Source of Data,56120,44947,11173,19.91
126 - Path Traversal,20950,16685,4265,20.36
34 - HTTP Response Splitting,19738,15823,3915,19.83
242 - Code Injection,15827,12617,3210,20.28
272 - Protocol Manipulation,9011,7207,1804,20.02
88 - OS Command Injection,7482,5928,1554,20.77
274 - HTTP Verb Tampering,5437,4305,1132,20.82
310 - Scanning for Vulnerable Software,2718,2184,534,19.65


## 7. Save Clean Split Artifacts

We serialize the partitions using `joblib` with level-3 compression for fast loading and reduced disk usage.

In [9]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(X_train, X_TRAIN_PATH, compress=3)
joblib.dump(X_test, X_TEST_PATH, compress=3)
joblib.dump(y_train, Y_TRAIN_PATH, compress=3)
joblib.dump(y_test, Y_TEST_PATH, compress=3)
joblib.dump(context_test, CONTEXT_TEST_PATH, compress=3)
context_test.to_csv(CONTEXT_TEST_CSV, index=True)

print("All split artifacts saved successfully to:", PROCESSED_DATA_DIR)

All split artifacts saved successfully to: C:\Users\Meishaaa\OneDrive - Sri Lanka Institute of Information Technology\Desktop updates\Desktop\RA 1\Dataset\processed
